**ALBERT DATA PREPROCESSING + BALANCING + TOKENIZATION**

In [ ]:
# ============================================================
# CELL 1
# ALBERT POLITICAL BIAS CLASSIFICATION
# DATA PREPROCESSING + TOKENIZATION
# ============================================================

# ============================================================
# INSTALLS
# ============================================================

# Uncomment if needed

# !pip install -q transformers datasets sentencepiece

# ============================================================
# IMPORTS
# ============================================================

import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample

from datasets import Dataset

from transformers import AutoTokenizer

# ============================================================
# SEED
# ============================================================

def seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)

seed_everything(42)

# ============================================================
# DEVICE
# ============================================================

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)

print("="*100)
print("DEVICE")
print("="*100)

print(f"\nUsing Device: {device}")

# ============================================================
# LOAD DATASET
# ============================================================

from pathlib import Path

# Finds the attached Kaggle dataset by file name, including ponssvvv/bias-clean.
_dataset_files = sorted(Path("/kaggle/input").rglob("bias_clean.csv"))
if not _dataset_files:
    raise FileNotFoundError("Attach the bias_clean Kaggle dataset (ponssvvv/bias-clean) in the notebook Input panel.")
DATA_PATH = str(_dataset_files[0])
print(f"Using dataset: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

# ============================================================
# CLEAN DATA
# ============================================================

df = df.dropna(

    subset=["page_text", "bias"]
)

df = df.reset_index(drop=True)

# ============================================================
# TEXT CLEANING
# ============================================================

def clean_text(text):

    text = str(text)

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

df["page_text"] = df["page_text"].apply(clean_text)

# ============================================================
# ORIGINAL DISTRIBUTION
# ============================================================

print("\n" + "="*100)
print("ORIGINAL LABEL DISTRIBUTION")
print("="*100)

print(df["bias"].value_counts())

# ============================================================
# BALANCE DATASET
# ============================================================

min_class_size = df["bias"].value_counts().min()

balanced_df = pd.DataFrame()

for label in df["bias"].unique():

    subset = df[df["bias"] == label]

    subset = resample(

        subset,

        replace=False,

        n_samples=min_class_size,

        random_state=42
    )

    balanced_df = pd.concat(

        [balanced_df, subset]
    )

balanced_df = balanced_df.sample(

    frac=1,

    random_state=42
).reset_index(drop=True)

# ============================================================
# LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

balanced_df["label"] = label_encoder.fit_transform(

    balanced_df["bias"]
)

# ============================================================
# LABEL MAPPING
# ============================================================

label_mapping = {

    idx: label

    for idx, label in enumerate(

        label_encoder.classes_
    )
}

print("\n" + "="*100)
print("LABEL MAPPING")
print("="*100)

print(label_mapping)

# ============================================================
# SPLIT DATA
# ============================================================

train_df, temp_df = train_test_split(

    balanced_df,

    test_size=0.30,

    stratify=balanced_df["label"],

    random_state=42
)

valid_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["label"],

    random_state=42
)

print("\n" + "="*100)
print("DATASET SPLITS")
print("="*100)

print(f"\nTrain Size      : {len(train_df)}")

print(f"Validation Size : {len(valid_df)}")

print(f"Test Size       : {len(test_df)}")

# ============================================================
# TOKENIZER
# ============================================================

MODEL_NAME = "albert-base-v2"

tokenizer = AutoTokenizer.from_pretrained(

    MODEL_NAME
)

MAX_LENGTH = 128

# ============================================================
# TOKENIZATION FUNCTION
# ============================================================

def tokenize_function(examples):

    return tokenizer(

        examples["page_text"],

        truncation=True,

        padding="max_length",

        max_length=MAX_LENGTH
    )

# ============================================================
# HF DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(

    train_df[["page_text", "label"]]
)

valid_dataset = Dataset.from_pandas(

    valid_df[["page_text", "label"]]
)

test_dataset = Dataset.from_pandas(

    test_df[["page_text", "label"]]
)

# ============================================================
# TOKENIZE
# ============================================================

train_dataset = train_dataset.map(

    tokenize_function,

    batched=True
)

valid_dataset = valid_dataset.map(

    tokenize_function,

    batched=True
)

test_dataset = test_dataset.map(

    tokenize_function,

    batched=True
)

# ============================================================
# FORMAT DATASETS
# ============================================================

train_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "label"
    ]
)

valid_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "label"
    ]
)

test_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "label"
    ]
)

# ============================================================
# LABEL DISTRIBUTION PLOT
# ============================================================

plt.figure(figsize=(10,6))

sns.countplot(

    x=balanced_df["bias"]
)

plt.title("Balanced Political Bias Distribution")

plt.xticks(rotation=20)

plt.show()

# ============================================================
# TEXT LENGTH DISTRIBUTION
# ============================================================

balanced_df["text_length"] = balanced_df["page_text"].apply(

    lambda x: len(str(x).split())
)

plt.figure(figsize=(10,6))

sns.histplot(

    balanced_df["text_length"],

    bins=50
)

plt.title("Text Length Distribution")

plt.xlabel("Word Count")

plt.show()

# ============================================================
# SAVE CSV FILES
# ============================================================

train_df.to_csv(

    "albert_train.csv",

    index=False
)

valid_df.to_csv(

    "albert_valid.csv",

    index=False
)

test_df.to_csv(

    "albert_test.csv",

    index=False
)

print("\n" + "="*100)
print("CELL 1 COMPLETED SUCCESSFULLY")
print("="*100)

print("""

COMPLETED:

✓ Dataset Cleaning
✓ Political Bias Balancing
✓ Label Encoding
✓ Train/Validation/Test Split
✓ ALBERT Tokenization
✓ Dataset Formatting
✓ Distribution Visualization
✓ Text Length Analysis

NEXT:

CELL 2 → ALBERT Training + Advanced Metrics

""")

**ALBERT TRAINING**

In [ ]:
# ============================================================
# CELL 2
# FINAL STABLE ALBERT TRAINING
# FAST + LIGHTWEIGHT + KAGGLE SAFE
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    matthews_corrcoef,

    cohen_kappa_score,

    confusion_matrix,

    classification_report
)

from transformers import (

    AutoModelForSequenceClassification,

    TrainingArguments,

    Trainer,

    EarlyStoppingCallback
)

# ============================================================
# LOAD MODEL
# ============================================================

print("="*100)
print("LOADING ALBERT MODEL")
print("="*100)

model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=len(label_mapping)
)

model.to(device)

print("\nModel Loaded Successfully.")

# ============================================================
# METRICS FUNCTION
# ============================================================

def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    accuracy = accuracy_score(

        labels,

        preds
    )

    precision = precision_score(

        labels,

        preds,

        average="weighted",

        zero_division=0
    )

    recall = recall_score(

        labels,

        preds,

        average="weighted",

        zero_division=0
    )

    f1 = f1_score(

        labels,

        preds,

        average="weighted",

        zero_division=0
    )

    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1
    }

# ============================================================
# TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir="./albert_results",

    num_train_epochs=40,

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    gradient_accumulation_steps=1,

    warmup_steps=100,

    weight_decay=0.01,

    max_grad_norm=1.0,

    logging_steps=100,

    save_steps=500,

    eval_steps=500,

    save_strategy="steps",

    eval_strategy="steps",

    do_train=True,

    do_eval=True,

    fp16=False,

    load_best_model_at_end=True,

    metric_for_best_model="f1"
)

# ============================================================
# TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    compute_metrics=compute_metrics,

    callbacks=[

        EarlyStoppingCallback(

            early_stopping_patience=4
        )
    ]
)

# ============================================================
# START TRAINING
# ============================================================

print("\n" + "="*100)
print("STARTING STABLE ALBERT TRAINING")
print("="*100)

trainer.train()

# ============================================================
# SAVE MODEL
# ============================================================

print("\n" + "="*100)
print("SAVING MODEL")
print("="*100)

trainer.save_model(

    "final_albert_bias_model"
)

tokenizer.save_pretrained(

    "final_albert_bias_model"
)

print("\nModel Saved Successfully.")

# ============================================================
# TEST EVALUATION
# ============================================================

print("\n" + "="*100)
print("RUNNING TEST EVALUATION")
print("="*100)

predictions = trainer.predict(

    test_dataset
)

preds = np.argmax(

    predictions.predictions,

    axis=1
)

labels = predictions.label_ids

# ============================================================
# ADVANCED METRICS
# ============================================================

accuracy = accuracy_score(

    labels,

    preds
)

precision = precision_score(

    labels,

    preds,

    average="weighted",

    zero_division=0
)

recall = recall_score(

    labels,

    preds,

    average="weighted",

    zero_division=0
)

weighted_f1 = f1_score(

    labels,

    preds,

    average="weighted",

    zero_division=0
)

macro_f1 = f1_score(

    labels,

    preds,

    average="macro",

    zero_division=0
)

mcc = matthews_corrcoef(

    labels,

    preds
)

kappa = cohen_kappa_score(

    labels,

    preds
)

# ============================================================
# PRINT METRICS
# ============================================================

print("\n" + "="*100)
print("ADVANCED EVALUATION METRICS")
print("="*100)

print(f"\nAccuracy           : {accuracy:.4f}")

print(f"Precision          : {precision:.4f}")

print(f"Recall             : {recall:.4f}")

print(f"Weighted F1        : {weighted_f1:.4f}")

print(f"Macro F1           : {macro_f1:.4f}")

print(f"Matthews Corrcoef  : {mcc:.4f}")

print(f"Cohen Kappa        : {kappa:.4f}")

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n" + "="*100)
print("CLASSIFICATION REPORT")
print("="*100)

print(

    classification_report(

        labels,

        preds,

        target_names=list(label_mapping.values()),

        zero_division=0
    )
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    labels,

    preds
)

plt.figure(figsize=(10,8))

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    cmap="Blues",

    xticklabels=label_mapping.values(),

    yticklabels=label_mapping.values()
)

plt.title("ALBERT Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

# ============================================================
# VALIDATION LOSS CURVE
# ============================================================

logs = trainer.state.log_history

eval_loss = []

epochs = []

for log in logs:

    if "eval_loss" in log:

        eval_loss.append(

            log["eval_loss"]
        )

        epochs.append(

            log["epoch"]
        )

plt.figure(figsize=(10,6))

plt.plot(

    epochs,

    eval_loss,

    marker='o',

    linewidth=2
)

plt.xlabel("Epoch")

plt.ylabel("Validation Loss")

plt.title("Validation Loss Curve")

plt.grid(True)

plt.show()

# ============================================================
# METRIC BAR PLOT
# ============================================================

metric_names = [

    "Accuracy",

    "Precision",

    "Recall",

    "Weighted F1",

    "Macro F1",

    "MCC",

    "Kappa"
]

metric_scores = [

    accuracy,

    precision,

    recall,

    weighted_f1,

    macro_f1,

    mcc,

    kappa
]

plt.figure(figsize=(12,6))

plt.bar(

    metric_names,

    metric_scores
)

plt.ylim(0,1)

plt.title("ALBERT Advanced Metrics")

plt.ylabel("Score")

plt.grid(axis='y')

plt.show()

# ============================================================
# PREDICTION DISTRIBUTION
# ============================================================

predicted_labels = [

    label_mapping[p]

    for p in preds
]

plt.figure(figsize=(10,6))

sns.countplot(

    x=predicted_labels
)

plt.title("Prediction Distribution")

plt.xticks(rotation=20)

plt.show()

# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

results_df = pd.DataFrame({

    "Metric": [

        "Accuracy",

        "Precision",

        "Recall",

        "Weighted F1",

        "Macro F1",

        "MCC",

        "Cohen Kappa"
    ],

    "Score": [

        accuracy,

        precision,

        recall,

        weighted_f1,

        macro_f1,

        mcc,

        kappa
    ]
})

print("\n" + "="*100)
print("FINAL SUMMARY")
print("="*100)

print(results_df)

# ============================================================
# FINAL SUCCESS
# ============================================================

print("\n" + "="*100)
print("CELL 2 COMPLETED SUCCESSFULLY")
print("="*100)

print("""

COMPLETED:

✓ Stable ALBERT Fine-Tuning
✓ Early Stopping
✓ Advanced Evaluation Metrics
✓ Accuracy Analysis
✓ Precision Analysis
✓ Recall Analysis
✓ Weighted F1
✓ Macro F1
✓ MCC
✓ Cohen Kappa
✓ Confusion Matrix
✓ Classification Report
✓ Validation Loss Curve
✓ Metric Visualization
✓ Prediction Distribution

NEXT:

CELL 3 → Advanced Curve Plots

""")

In [ ]:
# ============================================================
# CELL 3
# ALBERT ADVANCED CURVES + FINAL VISUALIZATION
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (

    roc_curve,

    auc,

    precision_recall_curve,

    confusion_matrix,

    accuracy_score
)

from sklearn.preprocessing import label_binarize

# ============================================================
# GET PROBABILITIES
# ============================================================

print("="*100)
print("GENERATING PREDICTION PROBABILITIES")
print("="*100)

probabilities = torch.softmax(

    torch.tensor(predictions.predictions),

    dim=1
).numpy()

# ============================================================
# ROC CURVES
# ============================================================

print("\n" + "="*100)
print("GENERATING ROC CURVES")
print("="*100)

y_true_bin = label_binarize(

    labels,

    classes=np.unique(labels)
)

n_classes = y_true_bin.shape[1]

plt.figure(figsize=(10,8))

for i in range(n_classes):

    fpr, tpr, _ = roc_curve(

        y_true_bin[:, i],

        probabilities[:, i]
    )

    roc_auc = auc(

        fpr,

        tpr
    )

    plt.plot(

        fpr,

        tpr,

        linewidth=2,

        label=f"Class {i} (AUC = {roc_auc:.2f})"
    )

plt.plot(

    [0,1],

    [0,1],

    linestyle='--'
)

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ALBERT Multi-Class ROC Curves")

plt.legend()

plt.grid(True)

plt.show()

# ============================================================
# PRECISION-RECALL CURVES
# ============================================================

print("\n" + "="*100)
print("GENERATING PRECISION-RECALL CURVES")
print("="*100)

plt.figure(figsize=(10,8))

for i in range(n_classes):

    precision_curve, recall_curve, _ = precision_recall_curve(

        y_true_bin[:, i],

        probabilities[:, i]
    )

    plt.plot(

        recall_curve,

        precision_curve,

        linewidth=2,

        label=f"Class {i}"
    )

plt.xlabel("Recall")

plt.ylabel("Precision")

plt.title("ALBERT Precision-Recall Curves")

plt.legend()

plt.grid(True)

plt.show()

# ============================================================
# CONFIDENCE SCORE DISTRIBUTION
# ============================================================

print("\n" + "="*100)
print("GENERATING CONFIDENCE DISTRIBUTION")
print("="*100)

confidence_scores = np.max(

    probabilities,

    axis=1
)

plt.figure(figsize=(10,6))

sns.histplot(

    confidence_scores,

    bins=30
)

plt.title("Prediction Confidence Distribution")

plt.xlabel("Confidence Score")

plt.ylabel("Frequency")

plt.show()

# ============================================================
# PREDICTION DISTRIBUTION
# ============================================================

print("\n" + "="*100)
print("GENERATING PREDICTION DISTRIBUTION")
print("="*100)

predicted_names = [

    label_mapping[p]

    for p in preds
]

plt.figure(figsize=(10,6))

sns.countplot(

    x=predicted_names
)

plt.title("Prediction Distribution")

plt.xticks(rotation=20)

plt.show()

# ============================================================
# PER-CLASS ACCURACY
# ============================================================

print("\n" + "="*100)
print("GENERATING PER-CLASS ACCURACY")
print("="*100)

per_class_accuracy = {}

for class_id in np.unique(labels):

    idxs = np.where(

        np.array(labels) == class_id
    )[0]

    acc = accuracy_score(

        np.array(labels)[idxs],

        np.array(preds)[idxs]
    )

    per_class_accuracy[

        label_mapping[class_id]
    ] = acc

# ============================================================
# PER-CLASS ACCURACY PLOT
# ============================================================

plt.figure(figsize=(10,6))

plt.bar(

    per_class_accuracy.keys(),

    per_class_accuracy.values()
)

plt.title("Per-Class Accuracy")

plt.ylabel("Accuracy")

plt.ylim(0,1)

plt.xticks(rotation=20)

plt.grid(axis='y')

plt.show()

# ============================================================
# VALIDATION LOSS CURVE
# ============================================================

print("\n" + "="*100)
print("GENERATING LOSS CURVE")
print("="*100)

logs = trainer.state.log_history

eval_loss = []

epochs = []

for log in logs:

    if "eval_loss" in log:

        eval_loss.append(

            log["eval_loss"]
        )

        epochs.append(

            log["epoch"]
        )

plt.figure(figsize=(10,6))

plt.plot(

    epochs,

    eval_loss,

    marker='o',

    linewidth=2
)

plt.xlabel("Epoch")

plt.ylabel("Validation Loss")

plt.title("ALBERT Validation Loss Curve")

plt.grid(True)

plt.show()

# ============================================================
# TRAINING LOSS CURVE
# ============================================================

train_loss = []

train_epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:

        train_loss.append(

            log["loss"]
        )

        train_epochs.append(

            log["epoch"]
        )

plt.figure(figsize=(10,6))

plt.plot(

    train_epochs,

    train_loss,

    marker='o',

    linewidth=2
)

plt.xlabel("Epoch")

plt.ylabel("Training Loss")

plt.title("ALBERT Training Loss Curve")

plt.grid(True)

plt.show()

# ============================================================
# OVERFITTING ANALYSIS
# ============================================================

plt.figure(figsize=(10,6))

plt.plot(

    train_epochs[:len(eval_loss)],

    train_loss[:len(eval_loss)],

    label="Training Loss",

    linewidth=2
)

plt.plot(

    epochs,

    eval_loss,

    label="Validation Loss",

    linewidth=2
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()

plt.grid(True)

plt.show()

# ============================================================
# FINAL SUCCESS
# ============================================================

print("\n" + "="*100)
print("CELL 3 COMPLETED SUCCESSFULLY")
print("="*100)

print("""

COMPLETED:

✓ ROC Curves
✓ Precision-Recall Curves
✓ Confidence Distribution
✓ Prediction Distribution
✓ Per-Class Accuracy
✓ Validation Loss Curve
✓ Training Loss Curve
✓ Overfitting Analysis
✓ Advanced Visualization

FINAL PIPELINE:

ALBERT Political Bias Classification System

""")

In [ ]:
# Reproducibility export: metrics and evaluation artifacts for the Kaggle output bundle.
from pathlib import Path
import json
import pandas as pd

_output_dir = Path("/kaggle/working/results")
_output_dir.mkdir(parents=True, exist_ok=True)
_metrics = {
    "model": 'ALBERT',
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "weighted_f1": float(weighted_f1),
    "macro_f1": float(macro_f1),
    "matthews_corrcoef": float(mcc),
    "cohen_kappa": float(kappa),
    "test_samples": int(len(test_df)),
}
(_output_dir / "albert_metrics.json").write_text(json.dumps(_metrics, indent=2), encoding="utf-8")
if "cm" in globals():
    pd.DataFrame(cm).to_csv(_output_dir / "albert_confusion_matrix.csv", index=False)
if "labels" in globals() and "preds" in globals():
    from sklearn.metrics import classification_report
    _names = list(label_mapping.values()) if "label_mapping" in globals() else None
    _report = classification_report(labels, preds, target_names=_names, output_dict=True, zero_division=0)
    pd.DataFrame(_report).transpose().to_csv(_output_dir / "albert_classification_report.csv")
print("Saved metrics and evaluation artifacts under /kaggle/working/results")